This notebook contains secondary calculations which back up our MacCabe Theil analysis performed.The cell below aims to compute the N_min viz. minimum number of stages inorder to achieve separation...our column should have more than or equal to N_min inorder to achieve separation effectively.

In [ ]:
try:
    import numpy as np
    from scipy.optimize import newton
    import matplotlib.pyplot as plt
    from thermo import Chemical
    
except ImportError as e:
    print(f"Error importing module: {e}. Please ensure all required libraries are installed.")
    exit(1)

# VLE data for n-hexane / cycloheptane at 1 atm
xy_data = {
    'x': [0.00,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,1.0],
    'y': [0.0, 0.1824197, 0.3218958, 0.430985, 0.5181176, 0.5890586, 0.6478281, 0.6972854, 0.7395021, 0.7760066,
          0.8079447, 0.8361889, 0.8614127, 0.8841429, 0.9047961, 0.9237058, 0.9411415, 0.9573229, 0.9724306, 0.9866146, 1.0]
}

# Convert to NumPy arrays
x_e = np.array(xy_data['x'])
y_e = np.array(xy_data['y'])
'''Chemical props of individual species for underwood method'''
chem1 = Chemical('n-hexane')
chem2 = Chemical('cycloheptane')


# Design specs (light key: n-hexane)
zF = 0.5   # feed composition
q = 1.0    # feed condition (1.0 = saturated liquid)
xD = 0.95
xB = 0.05
effeciency = 0.6    #this is the plate effeciency

def alpha(x_e, y_e):
    """Calculate relative volatility from equilibrium data (vectorized)."""
    # Avoid division by zero by masking out endpoints
    mask = (x_e > 0) & (x_e < 1) & (y_e > 0) & (y_e < 1)
    a_i = (y_e[mask] * (1 - x_e[mask])) / (x_e[mask] * (1 - y_e[mask]))
    return a_i

def geometric_ave(a_i):
    """Compute geometric mean of alpha."""
    return np.exp(np.mean(np.log(a_i)))

def fenske(alpha_avg, xD, xB):
    """Compute minimum number of stages from Fenske equation."""
    return np.log((xD / (1 - xD)) / (xB / (1 - xB))) / np.log(alpha_avg)

def underwood_method():
    '''
    Solves Underwood's equation for theta using Newton-Raphson,
    then calculates Rmin.
    '''
    
    # 1. Define the function for Underwood's first equation (f(theta) = 0)
    def f_theta(theta):
        # f(theta) = (alpha_LK * xD) / (alpha_LK - theta) + (alpha_HK * (1-xD)) / (alpha_HK - theta) - (1 - q)
        term1 = (alpha_ave * xD) / (alpha_ave - theta)
        term2 = (alpha_ave * (1 - xD)) / (alpha_ave - theta)
        return term1 + term2 - (1.0 - q)     #this is the first underwood eqn

    # 2. Define the derivative (f'(theta)) for faster/more stable Newton-Raphson
    def f_prime_theta(theta):
        # f'(theta) = d/d(theta) [f_theta]
        # d/d(theta) [ (A) / (B - theta) ] = (A) / (B - theta)^2
        term1_deriv = (alpha_ave * xD) / (alpha_ave - theta)**2
        term2_deriv = (alpha_ave* (1 - xD)) / (alpha_ave - theta)**2
        return term1_deriv + term2_deriv     #manually computd derivatives without Sympy
    
    # 3. Set a robust initial guess for theta
    # theta must be between the relative volatilities of the key components
    initial_guess_theta = 1.5
    
    # 4. Use scipy.optimize.newton to find the root (theta_solution)
    try:
        theta_solution = newton(
            func=f_theta, 
            x0=initial_guess_theta, 
            fprime=f_prime_theta, 
            tol=1e-6
        )
    except RuntimeError:
        print("Failed to converge to a solution for theta. Check initial guess or constants.")
        return None, None
    
    # 5. Calculate Rmin using the found theta_solution
    # Rmin + 1 = SUM [ alpha_i * x_D,i / (alpha_i - theta) ]
    Rmin_plus_1 = (alpha_ave * xD) / (alpha_ave - theta_solution) + \
                  (alpha_ave * (1 - xD)) / (alpha_ave - theta_solution)

    Rmin = Rmin_plus_1 - 1.0
    
    # 6. Return the results
    return Rmin, theta_solution

# Example Execution
Rmin_val, theta_val = underwood_method()

if Rmin_val is not None:
    print(f"Calculated Theta: {theta_val:.4f}")
    print(f"Calculated Rmin: {Rmin_val:.4f}")



def gilliland():
    """Placeholder for Gilliland method (not implemented)."""
    pass

# Main part
a_i = alpha(x_e, y_e)
alpha_ave = geometric_ave(a_i)

print("Volatility values for each data point:")
print(a_i)

print("\nGeometric average of volatility values (for Fenske):")
print(alpha_ave)

min_stages = fenske(alpha_ave, xD, xB)
actual_stages = min_stages/effeciency
print(f"\nMinimum number of stages from Fenske: Nmin = {min_stages:.2f}")
print(f"Actual number of stages (efficiency = {effeciency*100}%): N_actual = {actual_stages:.2f}")

#  Plot alpha vs composition to make it make sense
plt.plot(x_e[1:-1], a_i, 'o-', label='alphavs x')
plt.xlabel("Liquid mole fraction x")
plt.ylabel("Relative volatility ")
plt.title("Relative Volatility Profile from VLE Data")
plt.grid(True)
plt.legend()
plt.show()


NameError: name 'alpha_ave' is not defined

Now calculating minimum reflux ratio using the underwood equation.

In [ ]:
import numpy as np

# Known design specs
xD = 0.95   # distillate
xB = 0.05   # bottoms
xF = 0.50   # feed

# VLE data arrays (from earlier)
x_e = np.array([0.00,0.05,0.1,0.15,0.2,0.25,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,1.0])
y_e = np.array([0.0, 0.1824197, 0.3218958, 0.430985, 0.5181176, 0.5890586, 0.6478281, 0.6972854, 0.7395021,0.7760066,
                0.8079447, 0.8361889, 0.8614127, 0.8841429, 0.9047961, 0.9237058, 0.9411415, 0.9573229, 0.9724306, 
                0.9866146, 1.0])

# Step 1: Interpolate to get y_F (vapour composition in equilibrium with feed)
y_F = np.interp(xF, x_e, y_e)

# Step 2: Calculate R_min
R_min = (xD - y_F) / (y_F - xF)

print(f"Equilibrium y_F at x_F={xF}: {y_F:.4f}")
print(f"Minimum reflux ratio R_min: {R_min:.4f}")

R = 1.5 * R_min
print(f"recomended operating reflux ratio R (1.5 * R_min): {R:.4f}")

Equilibrium y_F at x_F=0.5: 0.8079
Minimum reflux ratio R_min: 0.4613
recomended operating reflux ratio R (1.5 * R_min): 0.6920
